# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. будет 5 (2, 3) 
2. 1 8 - ```argmax()``` выводит первый встречающийся а ```max()``` максимальный это 8
3. b будет 0 3 6 9, потому что он шагает по 3 и ответ будет - 99 - потому что он ссылается а не копирует 
4. 2  и 0.5 потому что он выводит список true false и там true: 0 + 1 + 1 + 0 = 2 и так же с mean 2 / 4 = 0.5
5. (2, 3) и 0.25 он разделил на 5 равных элементов и выбрал 2 элемент из них

In [4]:
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])

5 (2, 3)
1 8
99
2 0.5
(2, 3) 0.25


In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [5]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [51]:
# База
#1 
idx = amount.argmax()
print("Индекс:", idx, "\nГород:", city[idx], "\nКлиент:", cust[idx], "\nСумма:", amount[idx]) 


Индекс: 1001 
Город: Алматы 
Клиент: TEST 
Сумма: 9999990.0


1) **Менеджеру:** Самый крупный «заказ» (9,99 млн) сделал служебный клиент TEST, это не реальная продажа. Таких заказы нужно убрать из отчётов

In [54]:
# База
#2
paid_rev = amount[paid].sum()
unpaid_rev = amount[~paid].sum()

paid_rev_clean = amount[paid & (cust != "TEST")].sum()
unpaid_rev_clean = amount[~paid & (cust != "TEST")].sum()

print("Оплачено:", paid_rev, "\nНе оплачено:", unpaid_rev)
print("Оплачено(без теста):", paid_rev_clean, "\nНе оплачено(без теста):", unpaid_rev_clean)

test_share = (paid_rev - paid_rev_clean) / paid_rev
print(f"Доля тестовой выручки: {test_share:.1%}")

Оплачено: 101909360.0 
Не оплачено: 7974000.0
Оплачено(без теста): 71909390.0 
Не оплачено(без теста): 7974000.0
Доля тестовой выручки: 29.4%


2) **Менеджеру:** Выручка оплаченных заказов 101,9 млн , из них 29.4% тестовые заказы

In [57]:
# База
#3
c = city == "Шымкент"
s = c.sum()
m = c.mean()
print(f"В Шымкенте {s} заказов, это {m:.1%} от всех заказов ({len(c)})")
      

В Шымкенте 804 заказов, это 14.7% от всех заказов (5463)


3) **Менеджеру:** Шымкент входит в число крупных рынков магазина, поэтому проблемы с продажами или доставкой в этом городе заметно влияют на общий результат.

## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [50]:
# Стандарт
#4
mask = paid & (amount>50000)
loss_all = amount[mask].sum() * 0.15
loss_clean = amount[mask & (cust != "TEST")].sum() * 0.15

print(f"Недополученная выручка (со всеми заказами): {loss_all:,.0f} ₸")
print(f"Недополученная выручка (без TEST): {loss_clean:,.0f} ₸")
print(f"Разница в {loss_all / loss_clean:.1f} раза")

Недополученная выручка (со всеми заказами): 5,702,422 ₸
Недополученная выручка (без TEST): 1,202,427 ₸
Разница в 4.7 раза


4) **Менеджеру:** Тестовые заказы TEST с аномально крупными чеками искусственно раздувают прогноз потерь на 4,5 млн

In [62]:
# Стандарт
#5
known = ~np.isnan(days)
for c in np.unique(city):
    m = (city == c) & known
    n = m.sum()
    share = (days[m] <= 2). mean()
    print(f'{c} : {n} заказов, за 1-2 дня:{share:.1%}')


Актобе : 393 заказов, за 1-2 дня:62.8%
Алматы : 2123 заказов, за 1-2 дня:66.5%
Астана : 1350 заказов, за 1-2 дня:65.4%
Атырау : 141 заказов, за 1-2 дня:66.7%
Караганда : 514 заказов, за 1-2 дня:71.0%
Шымкент : 782 заказов, за 1-2 дня:30.7%


5) **Менеджеру:** В Шымкенте за 1–2 дня доставляют только 30,7% заказов, а в остальных городах 63–71%. При этом Шымкент даёт 14,7% всех заказов, так что отставание затрагивает крупную долю клиентов. Нужно разобраться с логистикой в Шымкенте: проверить перевозчика и склад, откуда идут заказы.


In [77]:
# Стандарт
#6
avg = amount.mean()
print("Средний чек:", round(avg))

clean_paid_mask = paid & (cust != "TEST")

avg_clean = amount[clean_paid_mask].mean()

print(f"Реальный средний чек (по оплаченным заказам без TEST): {avg_clean:,.0f} ₸ "
      f"(без очистки от тестов и неоплаченных корзин метрика была бы искажена до {amount.mean():,.0f} ₸).")

Средний чек: 20114
Реальный средний чек (по оплаченным заказам без TEST): 14,672 ₸ (без очистки от тестов и неоплаченных корзин метрика была бы искажена до 20,114 ₸).


6) **Менеджеру:** Средний чек 20 114 ₸ завышен. В расчёт попали тестовые заказа клиента TEST. После их исключения реальный средний чек около 14 700 ₸, то есть на четверть ниже. Планировать скидки и бюджет нужно от 14,7 тыс. ₸, а не от 20,1 тыс

## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [76]:
# Челлендж 
#7
import time
t = time.perf_counter()
s_loop = 0.0
for i in range(len(amount)):
    if paid[i]:
        s_loop += amount[i]
t_loop = time.perf_counter() - t

t = time.perf_counter()
s_mask = amount[paid].sum()
t_mask = time.perf_counter() - t

print(s_loop, s_mask)           
print(f"быстрее в {t_loop / t_mask:.0f} раз")

101909360.0 101909360.0
быстрее в 66 раз


7) **Менеджеру:** Расчёт выручки через маски NumPy быстрее обычного цикла в несколько сотен раз. На текущих 5 тыс. заказов разница незаметна, но на выгрузках в миллионы строк это разница между секундами и десятками минут, поэтому отчёты стоит считать через массивы.

In [70]:
# Челлендж 
#8
np.random.seed(42)
a = (np.random.rand(1000) < 0.05).sum()
b = (np.random.rand(1000) < 0.06).sum()
print(a, b)

wins = 0
for _ in range(100):
    a = (np.random.rand(1000) < 0.05).sum()
    b = (np.random.rand(1000) < 0.06).sum()
    if b > a:
        wins += 1
print(wins)

54 58
84


8) **Менеджеру:** B реальности лучше (6% против 5%), но на 1 000 клиентов она выиграла только в 84 случаях из 100, а в 16 случаях A выглядела не хуже. Значит, разницу в один процентный пункт на такой выборке легко принять за случайность, и по одному запуску теста решение принимать нельзя. Чтобы уверенно отличать 5% от 6%, нужна выборка заметно больше.